# Atlas-DeePC: Reacher Videos

Saved 300-step rollouts at **5x slower playback** (0.2x speed). Each video shows
the actual arm, one-step predicted pose, active reference posture, selected chart,
and target/reference/prediction errors. The experiment seed is in the video title.

All examples here use **p=8, K=64, rank=53**. Four successful examples were chosen
for targets in four different workspace quadrants and multiple chart visits;
they are illustrative selections, not a random performance sample.

The repair videos compare the **same seed, atlas, rank, initial history and target**.
The changed policy includes both graph waypoints and iterative terminal-reference
selection, so it is not an ablation of waypoints alone. Guidance does not guarantee
success: the full matched study recorded 36 repairs and 5 regressions in 500 pairs.

Viewing existing MP4s does not run the controller or require the experiment SSD.
Regenerating videos requires the saved traces on the SSD. Motion is interpolated
for display; all errors and outcomes come from saved measurements. Settled tails
are omitted: actual/reference/predicted arm points must remain within 1 mm of
their final values and chart IDs must stay fixed. We retain 10 extra steps,
round the cutoff up to 10 steps, and use a common cutoff for each matched pair.
Axes end at that cutoff, followed by a 1-second final hold. Original 300-step
data and full-rollout success/error statistics are unchanged.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Video, display

ROOT = next(root for parent in (Path.cwd(), *Path.cwd().parents)
            for root in (parent, parent / "code")
            if (root / "src/atlas_deepc").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
VIDEO_DIR = ROOT / "experiments/reacher/video_gallery_v1"
RENDER_VIDEOS = False  # Only renders saved traces; never launches control experiments.
OVERWRITE = False

if RENDER_VIDEOS:
    from atlas_deepc.reporting.reacher_video_gallery import render_gallery
    render_gallery(VIDEO_DIR, overwrite=OVERWRITE)

def show_video(filename):
    path = VIDEO_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Connect the SSD and enable RENDER_VIDEOS above.")
    # Embedded playback works in VS Code as well as Jupyter.
    display(Video(filename=str(path), embed=True, width=1100,
                  html_attributes="controls preload=metadata"))

manifest_path = VIDEO_DIR / "gallery.json"
if manifest_path.exists():
    gallery = json.loads(manifest_path.read_text())
    display(pd.DataFrame([
        {"seed": item["seed"], "mode": source["reference_mode"],
         "success": source["success"], "final100_mean_mm": source["final100_mean_mm"]}
        for item in gallery for source in item["sources"]
    ]))
print(f"Videos: {VIDEO_DIR}")


,seed,mode,success,final100_mean_mm
0,10339,graph,True,0.723329
1,10258,graph,True,3.751651
2,10215,graph,True,1.041288
3,10143,graph,True,0.965211
4,10206,terminal,False,377.344119
5,10206,graph,True,1.845311
6,10349,terminal,False,375.499437
7,10349,graph,True,0.424394
8,10445,terminal,False,191.072191
9,10445,graph,True,1.424071


Videos: /Users/kumiko/Projects/atlas_deepc/code/experiments/reacher/video_gallery_v1


## Reaching Different Targets

### Seed 10339: upper-right target
With graph guidance; p=8, K=64, rank=53.

In [2]:
show_video("target_seed10339_5x_slow.mp4")

### Seed 10258: upper-left target
With graph guidance; p=8, K=64, rank=53.

In [3]:
show_video("target_seed10258_5x_slow.mp4")

### Seed 10215: lower-left target
With graph guidance; p=8, K=64, rank=53.

In [4]:
show_video("target_seed10215_5x_slow.mp4")

### Seed 10143: lower-right target
With graph guidance; p=8, K=64, rank=53.

In [5]:
show_video("target_seed10143_5x_slow.mp4")

## Graph-Guidance Repairs

Left: without guidance, using a fixed terminal reference. Right: graph-guided
reference planning. The orange pose is the actual saved active reference, which
changes as the guided controller progresses through the charts. Both panels use
the same time and error scales. Final-100 errors refer to the complete rollout;
the live target error is shown separately.

The 10 mm line is a mean-error threshold, not an instantaneous success label.
Saved success requires both final-30 and final-100 mean errors below 10 mm and
maximum errors below 20 mm in those windows. Error axes are linear up to 10 mm
and logarithmic above it to show both initial and residual errors.


### Seed 10206: failure repaired by guidance

In [6]:
show_video("guidance_repair_seed10206_5x_slow.mp4")

### Seed 10349: failure repaired by guidance

In [7]:
show_video("guidance_repair_seed10349_5x_slow.mp4")

### Seed 10445: failure repaired by guidance

In [8]:
show_video("guidance_repair_seed10445_5x_slow.mp4")